# Time series: forecasting and anomaly detection

Three scenarios at increasing difficulty:

1. **Strong trend + seasonality** — linear trend model should win comfortably
2. **Noisy seasonal data** — neither simple model fits well → High headroom
3. **IoT sensor with anomalies** — both detection methods should agree


In [ ]:
import stepzero as sz
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


## Scenario 1 — Strong trend + seasonality (easy)

Clear upward trend and yearly cycle. `linear_trend` should beat `seasonal_naive` clearly.

In [ ]:
rng = np.random.default_rng(0)
idx = pd.date_range("2018-01", periods=60, freq="ME")
trend = np.linspace(100, 200, 60)
seasonal = 15 * np.sin(2 * np.pi * np.arange(60) / 12)
noise = rng.normal(0, 2, 60)
ts_easy = pd.Series(trend + seasonal + noise, index=idx, name="sales")

ts_easy.plot(title="Scenario 1: trend + seasonality", figsize=(10, 3))


In [ ]:
result = sz.forecasting(ts_easy, horizon=12)
print(result)
print()
print(result.headroom)
print()
for s in result.scores:
    print(f"  {s.name:<18} MAE = {s.score:.2f}")


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ts_easy.plot(ax=ax, label="observed")
result.forecast.plot(ax=ax, linestyle="--", marker="o", label=f"{result.best_model_name} forecast")
ax.legend()
ax.set_title("Scenario 1 — 12-month forecast")


## Scenario 2 — Noisy seasonal data (hard)

High noise swamps the seasonal signal. Neither simple model fits well.

In [ ]:
rng2 = np.random.default_rng(42)
idx2 = pd.date_range("2020-01", periods=36, freq="ME")
seasonal2 = 5 * np.sin(2 * np.pi * np.arange(36) / 12)
noise2 = rng2.normal(0, 20, 36)  # noise >> signal
ts_hard = pd.Series(50 + seasonal2 + noise2, index=idx2, name="demand")

ts_hard.plot(title="Scenario 2: high-noise seasonal", figsize=(10, 3))


In [ ]:
result2 = sz.forecasting(ts_hard, horizon=6)
print(result2)
print()
print(result2.headroom)


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ts_hard.plot(ax=ax, label="observed")
result2.forecast.plot(ax=ax, linestyle="--", marker="o", label=f"{result2.best_model_name} forecast")
ax.legend()
ax.set_title("Scenario 2 — high headroom: neither simple model is reliable")


## Scenario 3 — IoT sensor with anomalies

A temperature sensor running normally with occasional spikes and a slow drift.

In [ ]:
rng3 = np.random.default_rng(7)
t = np.arange(500)
normal_signal = 20 + 0.01 * t + rng3.normal(0, 0.5, 500)

# Inject anomalies: point spikes and a step change
anomaly_idx = [50, 150, 250, 350, 450]
normal_signal[anomaly_idx] += rng3.choice([-8, 8, 10, -10, 9])
# Brief step change
normal_signal[200:215] += 5

sensor = pd.Series(normal_signal, name="temperature_C")

fig, ax = plt.subplots(figsize=(12, 3))
sensor.plot(ax=ax, alpha=0.8)
ax.axvspan(200, 215, alpha=0.1, color="orange", label="step anomaly")
ax.set_title("IoT sensor signal")
ax.legend()


In [ ]:
result3 = sz.anomaly_detection(sensor)
print(result3)
print()
print(result3.headroom)


In [ ]:
# Compare z-score vs IQR side by side
result_z = sz.anomaly_detection(sensor, method="zscore")
result_iqr = sz.anomaly_detection(sensor, method="iqr")

fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)

for ax, res, title in zip(
    axes,
    [result3, result_z, result_iqr],
    [f"auto → {result3.method}", "z-score", "iqr"],
):
    sensor.plot(ax=ax, alpha=0.6, label="signal")
    sensor[res.anomalies].plot(ax=ax, style="rv", markersize=8, label="flagged")
    ax.set_title(f"{title}  (threshold={res.threshold:.2f}, flagged={res.anomalies.sum()})")
    ax.legend(loc="upper right")

plt.tight_layout()


### Takeaways

- **Scenario 1**: `linear_trend` beats `seasonal_naive` → headroom Low/Medium, simple model is production-ready
- **Scenario 2**: Both models struggle with high noise → headroom High, reach for ARIMA or Prophet
- **Scenario 3**: Z-score and IQR agree on clear point anomalies; the step change at t=200–215 may need a more sophisticated detector (e.g., Isolation Forest, CUSUM)
